# 07 · Visual audit

**Question.** Does every figure earn its place?

A table of figures against the encoding they use and the question they answer,
following `docs/standards/viz-guidelines.md`. This is the section that turns "we
made charts" into "each chart is a decision".

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
audit = pd.DataFrame([
    ("01 cause waffle", "area, labelled", "part-to-whole", "01_cause_waffle"),
    ("02 monthly composition", "stacked area", "composition over time", "02_monthly_composition"),
    ("03 carrier bars", "sorted bars, direct labels", "ranked comparison", "03_carrier_bars"),
    ("04 percentile band", "position with band", "distribution", "04_percentile_band"),
    ("05 lorenz", "position", "concentration", "05_lorenz"),
    ("06 distance bands", "position, flat lines", "near-null effect", "06_distance_bands"),
    ("07 state tiles", "colour + label", "geography, equal weight", "07_state_tiles"),
    ("08 cascade hour", "position + bar", "hypothetical turnaround pressure", "08_cascade_hour"),
    ("09 silhouette", "position", "selection evidence", "09_silhouette"),
    ("10 cluster scatter", "position + area", "similarity", "10_cluster_scatter"),
    ("11 importance", "length", "model explanation", "11_importance"),
    ("12 hour response", "position", "actionable risk", "12_hour_response"),
], columns=["figure", "encoding", "question", "file"])

audit["exists"] = [ (config.FIG_DIR / f"{name}.png").exists() for name in audit["file"] ]
audit[["figure", "encoding", "question", "exists"]]

## 7.1 What would be different with a weaker encoding

Three deliberate substitutions, recorded so the choice can be defended in the viva:

- A **bar of monthly means** replaced by a percentile band, because the mean sits
  above the median in every month.
- A **choropleth** relegated to second place behind an equal-area tile grid, because
  a choropleth gives large rural states visual weight unrelated to traffic.
- A **univariate R² bar chart** replaced by sequential incremental R², because the
  interesting number is the contribution after the other factors are in the model.

In [ ]:
missing = audit.loc[~audit["exists"], "figure"].tolist()
print("missing figures:", missing or "none - run python src/scripts/07_figures.py")

## Visual audit — Cairo's five questions, consolidated

The five questions are asked of every figure in the report. This table is the index;
the per-figure reasoning lives in the notebook that produces the figure.

| Figure | Truthful — how you can tell | Functional — how you can tell | Beautiful — how you can tell | Insightful — what it adds |
|---|---|---|---|---|
| 01 Cause waffle | Shares sum to 100% of attributed minutes; denominator printed; interval in the note | Family dominance readable in one pass; survives greyscale | Constant module size; finding in the title | Late aircraft outranks carrier, relocating the story to hand-over |
| 02 Monthly composition | 0–100% axis pinned, not auto-scaled; within-month shares | Late-aircraft band traceable without legend | Consistent hue order; thin separators | Mix tilts to inherited delay when volume peaks |
| 03 Carrier bars | Minutes per delayed flight, stated; thin-coverage carriers named | Sorting answers "who is worst"; values printed | Single fill; long names legible | Between-carrier spread is smaller than between-cause spread |
| 04 Percentile band | Median below zero and mean above it both shown | Month readable with median and 10–90 range | One hue, two tints, zero line emphasised | Reconciling "average" with "typical" before the text argues it |
| 05 Lorenz | Positive minutes only; Gini printed with the curve | 1% → 25%, 5% → 58.3% read directly | Two lines, one fill, dashed reference | Explains why the mean and median disagree |
| 06 Distance bands | Descriptive bands, different flight mixes flagged | Bands ordered by distance; sample sizes shown | Single encoding, no second axis | Distance barely moves delay once mix is acknowledged |
| 07 State tiles | Territory codes included and said so | Colour ramp with legend and labelled extremes | Ramp chosen for monotone luminance | Geography is a weaker split than hour or month |
| 08 Cascade hour | "Hypothetical pressure" in the title; interval stated | Peak hour locatable; sparse hours drawn thin | One line, one band, one annotation | Pressure peaks where modelled risk peaks |
| 09 Silhouette | Chosen k marked; stability verdict printed | Constraint drawn, not described | One line, one marker | Separation is not reproducibility |
| 10 Cluster scatter | Axes named; labels describe regions, not types | Named airport findable | Four hues, size legend off the points | Hub scale does not separate the groups |
| 11 Importance | Permutation importance, not SHAP; non-additive note | Rank order readable; small factors kept | Sorted bars, light error bars | Strongest driver is already on the timetable |
| 12 Hour response | Base rate drawn; split dates in the note | Hour-level risk readable against the rule | Natural 0–23 order | Risk triples from morning to afternoon |
| 13 Dashboard overview | Labelled as a figure overview, not a live screenshot | Each panel identifiable by its own title | Consistent panel grid | Shows the product surface, not evidence |
| 14 Mean vs band pair | Same twelve aggregates in both panels | Reader can verify the claim inside one page | Identical styling isolates the encoding | The audit changed a chart, not just its caption |
| 15 Temporal validation | Test years and folds named; negative Brier gain shown | Year-by-year comparison readable | One panel per model family | Signal weakens out of sample, and 2020 breaks the calibration |
| 16 Inference evidence | Coefficients with clustered intervals, reference levels named | Sign and magnitude readable per factor | Forest layout, zero rule | Average effects are small and mostly hourly |

### What the audit changed

Three decisions came out of asking these questions rather than out of taste:

1. The monthly composition chart moved from an auto-scaled axis to a pinned 0–100%
   axis, because the auto-scaled version made a two-point wobble look like a regime
   change.
2. The mean-per-month chart was replaced by the paired encoding in Figure 14, because
   a single mean line implied a typical flight that the median contradicts.
3. The cluster figure stopped labelling groups as airport *types* and started
   labelling them as regions of feature space, because the stability test (ARI 0.7562,
   minimum 0.4747) does not support a typology.
